# Minimum Current Estimate (MCE) methods

In [32]:
# import
import mindmap as mm

# reader
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("simf10")

# data
fwd = sr.load("fwd")
ave = sr.load("ave", task="rest", run="11")
ncov = sr.load("cov", task="rest", run="11")
eig = sr.load("eig")

ave = ave.crop(tmin=-0.1, tmax=0.1)

In [ ]:
import mne
import numpy as np
from mne.proj import make_projector
from mne.cov import prepare_noise_cov, compute_whitener
from mindmap.analysis.functional.inverse.utils import compute_depth_prior
import scipy.linalg as sla
from mindmap.analysis.functional.inverse.simulation import make_stc

import pyxu.abc as pxa
import pyxu.operator as pxo
import pyxu.opt.stop as pxst
from pyxu.opt.solver import PGD

class MCE:
    """Base inverse solver for MCE problems."""
    def __init__(
        self,
        forward: mne.Forward,
        noise_cov: mne.Covariance,
        depth: float = 0.8,
        verbose: str = "warning",
    ):
        """
        Base class for M/EEG inverse operators.
        """
        # input data
        self.forward = mne.convert_forward_solution(forward, force_fixed=True, verbose=verbose)
        self.noise_cov = noise_cov
        self.gain_raw = self.forward["sol"]["data"]

        # parameters
        self.depth_coeff = depth
        
        # variable properties
        self.lam = None

        # others
        self.verbose = verbose


    def _setup_preprocessor(self, info):
        """Extract whitener from noise covariance."""
        # projectors
        projs_gain, _, _ = make_projector(info["projs"], info.ch_names)
        projs_data, _, _ = make_projector(info["projs"], info.ch_names, include_active=False)

        # whitener
        ready_cov = prepare_noise_cov(self.noise_cov, info, verbose=self.verbose)
        whitener, _ = compute_whitener(self.noise_cov, info, verbose=self.verbose)

        # store rank-related info
        self.rank = np.sum(ready_cov["eig"]>0)

        return projs_gain, projs_data, whitener

    def compute_source_prior(self, gain_whitened):
        """Default: standard depth weighting."""
        return compute_depth_prior(gain_whitened, self.depth_coeff)
    
    def prepare_kernel(self, projs, whitener):
        """Builds kernel for source reconstruction"""
        # 1. Projectors and whitener
        gain_white = whitener @ projs @ self.gain_raw

        # 2. Get the prior
        # This returns either a 1D vector (diagonal) or 2D matrix
        w_source = self.compute_source_prior(gain_white)

        # 3. Apply prior and SVD
        if w_source.ndim == 1:
            gain_weighted = gain_white * w_source
        else:
            gain_weighted = gain_white @ w_source

        # 4. Scale source variance (to match EEG noise variance range)
        trace_GRGT = np.linalg.norm(gain_weighted, ord="fro")**2
        scale = np.sqrt(self.rank/trace_GRGT)

        gain_scaled = gain_weighted * scale
        w_source_scaled = w_source * scale

        return {
            "gain": gain_scaled,
            "w_source": w_source_scaled,
        }
    
    def estimate_alpha_max(self, y_white, gain_weighted):
        """
        Calculates the alpha value at which all coefficients become zero.
        Alpha_max = max(||G.T @ M||_2 over rows)
        """
        corr = gain_weighted.T @ y_white

        # For L21, alpha_max is the maximum L2-norm of the rows of the correlation
        alpha_max = np.max(np.linalg.norm(corr, axis=1))
        return alpha_max

    def solve(self, evoked, alpha=0.2, maxiter=1000, rel_tol=1e-4):
        """Solve inverse problem"""
        # Preprocessors
        projs_gain, projs_data, whitener = self._setup_preprocessor(evoked.info)

        # 1. Process gain
        kern = self.prepare_kernel(projs_gain, whitener)

        # 2. Process data
        y_white = whitener @ projs_data @ evoked.data

        # 3. Regularizer
        a_max = self.estimate_alpha_max(y_white, kern["gain"])
        self.lam = alpha * a_max
        
        # 3. Inverse logic (using pyxu)

        n_chan, n_sources = kern["gain"].shape
        n_times = len(evoked.times)

        # pyxu operators
        gain = pxa.LinOp.from_array(kern["gain"])
        eye = pxo.IdentityOp(dim_shape=n_times)
        y_vec = y_white.flatten()
        loss = pxo.SquaredL2Norm(dim_shape=y_vec.size).argshift(-y_vec) @ pxo.kron(eye, gain)
        
        prior = alpha * pxo.L21Norm(dim_shape=(n_sources, n_times), l2_axis=1)

        loss.diff_lipschitz = loss.estimate_diff_lipschitz()

        stop_crit = pxst.RelError(eps=rel_tol, var="x") | pxst.MaxIter(maxiter)
        solver = PGD(f=loss, g=prior, show_progress=False)

        x0 = np.zeros((n_sources, n_times)).flatten()
        solver.fit(x0=x0, stop_crit=stop_crit)

        sol = solver.solution().reshape(n_sources, n_times)

        # 4. Map back with prior
        if kern["w_source"].ndim == 1:
            sol = sol * kern["w_source"][:, np.newaxis]
        else:
            sol = kern["w_source"] @ sol

        return make_stc(
            source_space = self.forward["src"],
            activity = sol,
            tmin = evoked.times[0],
            tstep = 1/evoked.info["sfreq"]
        )



In [ ]:
test = MCE(fwd, ncov, 0.8)
stc = test.solve(ave)

AttributeError: 'SquareOp' object has no attribute 'shape'

In [72]:
# Preprocessors
projs_gain, projs_data, whitener = test._setup_preprocessor(ave.info)

# 1. Process gain
kern = test.prepare_kernel(projs_gain, whitener)

# 2. Process data
y_white = whitener @ projs_data @ ave.data

# 3. Regularizer
a_max = test.estimate_alpha_max(y_white, kern["gain"])
lam = 0.2 * a_max
        
# 3. Inverse logic (using pyxu)

n_chan, n_sources = kern["gain"].shape
n_times = len(ave.times)

# pyxu operators
gain = pxa.LinOp.from_array(kern["gain"])
gain.shape = (gain.codim_size, gain.dim_size)
eye = pxo.IdentityOp(dim_shape=n_times)
eye.shape = (eye.codim_size, eye.dim_size)
y_vec = y_white.flatten()
loss = pxo.SquaredL2Norm(dim_shape=y_vec.size).argshift(-y_vec) @ pxo.kron(eye, gain)
        
prior = 0.2 * pxo.L21Norm(dim_shape=(n_sources, n_times), l2_axis=1)

loss.diff_lipschitz = loss.estimate_diff_lipschitz()**2

stop_crit = pxst.RelError(eps=1e-3, var="x") | pxst.MaxIter(1000)
solver = PGD(f=loss, g=prior, show_progress=False)

x0 = np.zeros((n_sources, n_times)).flatten()
solver.fit(x0=x0, stop_crit=stop_crit)

sol = solver.solution().reshape(n_sources, n_times)

# 4. Map back with prior
if kern["w_source"].ndim == 1:
    sol = sol * kern["w_source"][:, np.newaxis]
else:
    sol = kern["w_source"] @ sol

make_stc(
    source_space = fwd["src"],
    activity = sol,
    tmin = ave.times[0],
    tstep = 1/ave.info["sfreq"]
)

AttributeError: 'SquareOp' object has no attribute 'shape'

'2.0.3'

In [ ]:
import numpy as np
import pyxu.abc as pxa
import pyxu.operator as pxo

ones = pxa.LinOp.from_array(np.ones((2,2)))
eye = pxo.IdentityOp(dim_shape=2)
kron = pxo.kron(eye, ones)


AttributeError: 'SquareOp' object has no attribute 'shape'

In [56]:
mm.viz.plot_source_estimate(fwd["src"], stc, ts_params={"absolute":True, "normalize":"robust"}).show()

Widget(value='<iframe src="http://localhost:43431/index.html?ui=P_0x738aa30e6b50_3&reconnect=auto" class="pyvi…

In [ ]:
mm.viz.plot_brain(fwd["src"], )

np.float64(1.7097619958043407e-09)

In [53]:
sol.max()

np.float64(1.7106834286594135e-09)